In [ ]:
# ==============================================================================
# CELL 0: MULTI-MODEL ABSORPTION COMPARISON (NEW OPTIMIZED FILES)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# --- 0. STYLE CONFIGURATION ---
def matplotlib_default_config():
    """Returns the default dictionary configurations for publication-ready plots."""
    font = {'family' : 'serif', 'weight' : 'normal', 'size'   : 14}
    rcnew = {
        "mathtext.fontset" : "cm", 
        "text.usetex": False,
        'figure.frameon': True,
        'axes.linewidth': 2.,
        "axes.titlesize" : 18,
        "axes.labelsize" : 18,
        "legend.fontsize" : 14,
        'legend.fancybox': True,
        'lines.linewidth': 2.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.top': True,
        'ytick.right': True,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16
    }
    return font, rcnew
    
# Apply Config
%matplotlib widget 
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

# --- 1. PARAMETERS & NEW FILE PATHS ---
# Dictionary pointing to the newly extracted batch files
DATA_FILES = {
    'Fiducial': '/home/vroberto/FakeSpectra_DLAs/data/fiducial_AllHalos_DLAMode_FullMap_15.pkl',
    'LH_0':     '/home/vroberto/FakeSpectra_DLAs/data/LH_0_AllHalos_DLAMode_FullMap_15.pkl',
    'LH_1':     '/home/vroberto/FakeSpectra_DLAs/data/LH_1_AllHalos_DLAMode_FullMap_15.pkl'
}

H_HUBBLE = 0.6774 

# Define the target mass bin
TARGET_MASS_MIN = 11.0
TARGET_MASS_MAX = 11.4

# Base colors for the comparison
COLORS = {
    'Fiducial': '#1f4e78', # Strong Blue
    'LH_0': '#b30000',     # Strong Red
    'LH_1': '#006400'      # Dark Green
}
LINESTYLES = {
    'Fiducial': '-',
    'LH_0': '--',
    'LH_1': '-.'
}

# --- 2. DATA LOADING & PREP ---
dfs_filtered = {}
available_radii = None

print(f"--> Filtering halos with mass between {TARGET_MASS_MIN} and {TARGET_MASS_MAX} log10(M_sun)")

for model, path in DATA_FILES.items():
    if not os.path.exists(path):
        print(f"ERROR: {path} not found. Skipping {model}.")
        continue
        
    print(f"Loading {model}...")
    df = pd.read_pickle(path)
    
    # SAFETY FIX: Drop zero or negative masses before log10 to prevent ValueError/NaN crashes
    df = df[df['halo_mass'] > 0].copy()
    
    # Calculate log_mass_msun
    df['log_mass_msun'] = np.log10((df['halo_mass'] * 1e10) / H_HUBBLE)
    
    # Filter DataFrames for the target mass bin ONLY
    mask = (df['log_mass_msun'] >= TARGET_MASS_MIN) & (df['log_mass_msun'] < TARGET_MASS_MAX)
    df_filtered = df[mask].copy()
    
    # Round r_multiplier to avoid float precision grouping issues
    df_filtered['r_mult_round'] = df_filtered['r_multiplier'].round(2)
    dfs_filtered[model] = df_filtered
    
    # Identify impact parameters based on the first successfully loaded model
    if available_radii is None and not df_filtered.empty:
        available_radii = np.sort(df_filtered['r_mult_round'].unique())

if available_radii is None:
    print("ERROR: No data found in the specified mass bin across all models.")
    import sys; sys.exit()

NUM_RADII = len(available_radii)
print("\n--- Halos Count per Radius Bin (Mass Filtered) ---")

# --- 3. PLOTTING ---
# Create stacked panels, one for each impact radius
fig, axes = plt.subplots(NUM_RADII, 1, figsize=(12, 4 * NUM_RADII), sharex=True)
if NUM_RADII == 1: axes = [axes]

# Common interpolation grid (Wide enough to see the absorption wings)
wave_grid = np.linspace(1215.67 - 10, 1215.67 + 10, 600)

for ax_idx, r_target in enumerate(available_radii):
    ax = axes[ax_idx]
    counts_text = []
    
    for model in DATA_FILES.keys():
        if model not in dfs_filtered: continue
            
        df_model = dfs_filtered[model]
        bin_data = df_model[df_model['r_mult_round'] == r_target]
        
        # Count unique halos in this specific mass + radius bin
        num_halos = bin_data['halo_id'].nunique()
        counts_text.append(f"{model}: {num_halos}")
        
        if bin_data.empty: continue
            
        # Interpolate flux onto common grid
        flux_mat = np.array([np.interp(wave_grid, row['wave'], row['flux']) for _, row in bin_data.iterrows()])
        mean_flux = np.nanmean(flux_mat, axis=0)
        std_flux = np.nanstd(flux_mat, axis=0)
        
        # Plot individual spectra (highly transparent)
        for f in flux_mat:
            ax.plot(wave_grid, f, color=COLORS[model], alpha=0.015, lw=0.5, zorder=1, rasterized=True)
            
        # Plot mean line and shaded standard deviation
        lbl = f"{model} Mean" if ax_idx == 0 else None
        ax.plot(wave_grid, mean_flux, color=COLORS[model], lw=2.5, linestyle=LINESTYLES[model], label=lbl, zorder=10)
        ax.fill_between(wave_grid, mean_flux - std_flux, mean_flux + std_flux, color=COLORS[model], alpha=0.2, zorder=5)

    # Set subplot title explicitly mentioning the mass bin
    title_str = f"Impact Parameter: $b \\approx {r_target:.2f} R_{{200}}$ | Mass Bin: $[10^{{{TARGET_MASS_MIN}}}, 10^{{{TARGET_MASS_MAX}}}) \\ M_\\odot$"
    ax.set_title(title_str, pad=10)
    
    # Add Halo Count Text Box to Plot
    text_str = "Halos in Mass Bin:\n" + "\n".join(counts_text)
    props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
    ax.text(0.02, 0.15, text_str, transform=ax.transAxes, fontsize=12,
            verticalalignment='bottom', bbox=props, zorder=20)

    # Reference Lines & Limits
    ax.axhline(1.0, color='gray', linestyle=':', lw=1.5, zorder=0)
    ax.axhline(0.0, color='gray', linestyle=':', lw=1.5, zorder=0)
    ax.axvline(1215.67, color='black', linestyle='-', lw=1.0, alpha=0.5, zorder=0)

    ax.set_ylim(-0.05, 1.1)
    ax.set_ylabel(r'$F/F_0$')
    
    # Legend only on the first panel
    if ax_idx == 0:
        ax.legend(loc='lower right', framealpha=0.9, fontsize=14)

print("------------------------------------------\n")

# Format the X-axis only on the bottom panel
axes[-1].set_xlabel(r'Wavelength $\lambda\ [\mathrm{\AA}]$')
fig.subplots_adjust(hspace=0.15)



plt.show()

--> Filtering halos with mass between 11.0 and 11.4 log10(M_sun)
Loading Fiducial...
Loading LH_0...
Loading LH_1...

--- Halos Count per Radius Bin (Mass Filtered) ---
